# Healthcare Analytics Demo — One-Click Launcher

**Run All** to deploy the complete Healthcare Payer/Provider analytics solution into this Fabric workspace.

### What Gets Deployed
| Layer | Items |
|-------|-------|
| **Lakehouses** | `lh_bronze_raw`, `lh_silver_stage`, `lh_silver_ods`, `lh_gold_curated` |
| **Notebooks** | 5 ETL + 2 data generation + 5 RTI (event simulator, Eventhouse setup, 3 scoring) |
| **Pipelines** | `PL_Healthcare_Full_Load`, `PL_Healthcare_Master` |
| **Semantic Model** | `HealthcareDemoHLS` (star schema) |
| **Data Agent** | `HealthcareHLSAgent` (Copilot AI) |
| **Ontology** | `Healthcare_Demo_Ontology_HLS` — **manual UI setup** (guide provided in Cell 7) |
| **Real-Time Intelligence** | Eventhouse + KQL DB + 3 scoring use cases (fraud, care gaps, high-cost trajectory) |

### Prerequisites
- An **empty** Fabric workspace (this notebook should be the only item)
- Fabric capacity: **F64** or higher recommended
- User must be workspace **Admin** or **Member**

### Configuration
Edit the cell below to point to your GitHub repo (public or private).

In [ ]:
# ============================================================================
# CONFIGURATION — Edit these values
# ============================================================================

GITHUB_OWNER = "rasgiza"                # GitHub org or user (public mirror)
GITHUB_REPO  = "Fabric-Payer-Provider-HealthCare-Demo"
GITHUB_BRANCH = "main"
GITHUB_TOKEN = ""                    # Leave empty for public repos. Only needed for private repos (classic PAT with repo scope).

# Deployment options
GENERATE_DATA = True                  # Generate fresh synthetic data
RUN_PIPELINE = True                   # Run the full-load pipeline after data gen
UPLOAD_KNOWLEDGE_DOCS = True          # Upload healthcare knowledge docs to lakehouse
DEPLOY_RTI = True                     # Deploy Real-Time Intelligence (Eventhouse + scoring)

print(f"Will deploy from: github.com/{GITHUB_OWNER}/{GITHUB_REPO} @ {GITHUB_BRANCH}")

In [ ]:
# ============================================================================
# CELL 1 — Install fabric-launcher
# ============================================================================

%pip install fabric-launcher --quiet

import notebookutils
from fabric_launcher import FabricLauncher

print("fabric-launcher installed successfully")

In [ ]:
# ============================================================================
# CELL 2 — Initialize launcher
# ============================================================================

launcher = FabricLauncher(
    notebookutils,
    environment="DEV",
    allow_non_empty_workspace=False,
    fix_zero_logical_ids=True,
    debug=False,
)

workspace_id = notebookutils.runtime.context.get("currentWorkspaceId", "")
print(f"Workspace ID: {workspace_id}")
print(f"Launcher ready")

In [ ]:
# ============================================================================
# CELL 3 — Download repo & deploy all Fabric artifacts from scratch (staged)
# ============================================================================
# Downloads the repo as a ZIP, extracts it, and creates every artifact
# in the workspace using the Fabric REST API.  Order matters because
# later items reference earlier ones via logicalId.
#
# Stage 1: Lakehouses          — must exist before notebooks reference them
# Stage 2: Eventhouse           — async provisioning, needs its own stage
# Stage 3: KQL Database         — depends on Eventhouse being fully ready
# Stage 4: Notebooks (create)  — creates notebook items in workspace
# Stage 5: Notebooks (content) — updateDefinition applies code reliably
#           (Fabric Create Item API can silently drop inline definitions
#            for notebooks; a second pass forces updateDefinition which
#            always applies the content.)
# Stage 6: Data Pipelines      — orchestrate notebook execution
# Stage 7: Semantic Model + Data Agent — need Gold tables populated first
# ============================================================================

print("Downloading repo and deploying artifacts...")
print("This takes 3-5 minutes.\n")

downloader, deployer, report = launcher.download_and_deploy(
    repo_owner=GITHUB_OWNER,
    repo_name=GITHUB_REPO,
    branch=GITHUB_BRANCH,
    github_token=GITHUB_TOKEN if GITHUB_TOKEN else None,
    workspace_folder="workspace",
    item_type_stages=[
        ["Lakehouse"],                         # Stage 1
        ["Eventhouse"],                        # Stage 2 — provision Eventhouse first
        ["KQLDatabase"],                       # Stage 3 — KQL DB needs Eventhouse ready
        ["Notebook"],                          # Stage 4 — create notebook items
        ["Notebook"],                          # Stage 5 — updateDefinition ensures code is applied
        ["DataPipeline"],                      # Stage 6
        ["SemanticModel", "DataAgent"],        # Stage 7
    ],
    validate_after_deployment=True,
    generate_report=True,
    deployment_retries=2,
)

print("\n" + "=" * 60)
print("  ARTIFACT DEPLOYMENT COMPLETE")
print("=" * 60)

In [ ]:
# ============================================================================
# CELL 3b — Ensure notebook content is applied via REST API
# ============================================================================
# The Fabric Create Item API can silently create notebook shells without
# code. This cell reads each notebook definition from the extracted repo
# and pushes it directly via the updateDefinition REST API.
#
# Strategy: Include ALL files (.platform + metadata + content) and use
# updateMetadata=True — this matches exactly what fabric_cicd does for
# existing items. Then verify by calling getDefinition on each notebook.
# ============================================================================

import base64, requests, os, json, time

print("Pushing notebook content via REST API (updateDefinition)...\n")

token = notebookutils.credentials.getToken("pbi")
hdrs = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
api = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}"

# 1 — Get all deployed items for name→ID mapping
resp = requests.get(f"{api}/items", headers=hdrs)
all_items = resp.json().get("value", [])
name_to_id = {(it["type"], it["displayName"]): it["id"] for it in all_items}

# 2 — Locate extracted workspace directory
ws_dir = None
for candidate in [".lakehouse/default/Files/src", "/lakehouse/default/Files/src"]:
    d = os.path.join(candidate, "workspace")
    if os.path.isdir(d):
        ws_dir = d
        break

if not ws_dir:
    print("ERROR: Could not find extracted workspace directory. Cannot push notebooks.")
else:
    # 3 — Build logical_id → real_id mapping from ALL .platform files
    logical_to_real = {}
    for entry in os.listdir(ws_dir):
        pf = os.path.join(ws_dir, entry, ".platform")
        if os.path.exists(pf):
            with open(pf, "r") as f:
                plat = json.load(f)
            itype = plat["metadata"]["type"]
            iname = plat["metadata"]["displayName"]
            lid = plat["config"]["logicalId"]
            rid = name_to_id.get((itype, iname), "")
            if lid and rid:
                logical_to_real[lid] = rid

    # 4 — Push each notebook via updateDefinition (include .platform)
    pushed, failed = 0, 0
    for entry in sorted(os.listdir(ws_dir)):
        if not entry.endswith(".Notebook"):
            continue
        nb_name = entry.replace(".Notebook", "")
        nb_id = name_to_id.get(("Notebook", nb_name))
        if not nb_id:
            print(f"  SKIP {nb_name}: not found in workspace")
            continue

        nb_path = os.path.join(ws_dir, entry)
        parts = []
        for root, _dirs, files_in_dir in os.walk(nb_path):
            for fname in files_in_dir:
                fpath = os.path.join(root, fname)
                relpath = os.path.relpath(fpath, nb_path).replace("\\", "/")

                # .platform — include as-is (no replacements), same as fabric_cicd
                if fname == ".platform":
                    with open(fpath, "r", encoding="utf-8") as f:
                        raw = f.read()
                    encoded = base64.b64encode(raw.encode("utf-8")).decode("utf-8")
                    parts.append({"path": relpath, "payload": encoded, "payloadType": "InlineBase64"})
                    continue

                # All other text files — apply ID and workspace replacements
                with open(fpath, "r", encoding="utf-8") as f:
                    content = f.read()
                for lid, rid in logical_to_real.items():
                    content = content.replace(lid, rid)
                content = content.replace("00000000-0000-0000-0000-000000000000", workspace_id)
                encoded = base64.b64encode(content.encode("utf-8")).decode("utf-8")
                parts.append({"path": relpath, "payload": encoded, "payloadType": "InlineBase64"})

        if not parts:
            print(f"  SKIP {nb_name}: no files found")
            continue

        # POST updateDefinition with updateMetadata=True (requires .platform)
        body = {"definition": {"parts": parts}}
        r = requests.post(
            f"{api}/items/{nb_id}/updateDefinition?updateMetadata=True",
            headers=hdrs,
            json=body,
        )

        # Handle long-running operation (202)
        if r.status_code == 202:
            loc = r.headers.get("Location", "")
            retry_after = int(r.headers.get("Retry-After", "2"))
            if loc:
                for attempt in range(60):
                    time.sleep(retry_after)
                    pr = requests.get(loc, headers=hdrs)
                    if pr.status_code == 200:
                        pdata = pr.json() if pr.text else {}
                        op_status = pdata.get("status", "Succeeded")
                        if op_status in ("Succeeded", "Running"):
                            if op_status == "Succeeded":
                                break
                        elif op_status == "Failed":
                            print(f"  FAIL {nb_name}: LRO failed — {pdata.get('error', {}).get('message', 'unknown')}")
                            failed += 1
                            break
                    elif pr.status_code == 202:
                        continue  # still in progress
                else:
                    print(f"  WARN {nb_name}: LRO timed out after polling")

        if r.status_code in (200, 202):
            pushed += 1
            print(f"  {nb_name}: OK ({len(parts)} parts)")
        else:
            failed += 1
            msg = r.text[:300] if r.text else "no response body"
            print(f"  FAIL {nb_name}: HTTP {r.status_code} — {msg}")

    print(f"\n{'='*60}")
    print(f"  Notebooks updated: {pushed}")
    if failed:
        print(f"  Failed: {failed}")

    # 5 — Verify by reading back one notebook definition
    print(f"\nVerifying notebook content...")
    test_nb = "01_Bronze_Ingest_CSV"
    test_id = name_to_id.get(("Notebook", test_nb))
    if test_id:
        vr = requests.post(f"{api}/items/{test_id}/getDefinition", headers=hdrs)
        if vr.status_code == 200:
            vdata = vr.json()
            defn_parts = vdata.get("definition", {}).get("parts", [])
            content_part = next((p for p in defn_parts if "notebook-content" in p.get("path", "")), None)
            if content_part:
                decoded = base64.b64decode(content_part["payload"]).decode("utf-8")
                line_count = decoded.count("\n")
                has_code = "spark" in decoded.lower() or "pyspark" in decoded.lower()
                print(f"  {test_nb}: {line_count} lines, has_code={has_code}")
                if line_count < 10:
                    print(f"  WARNING: Notebook appears to have very little content!")
                    print(f"  First 200 chars: {decoded[:200]}")
                else:
                    print(f"  Verified OK — notebook has real code content")
            else:
                print(f"  WARNING: No notebook-content part found in definition!")
                print(f"  Parts: {[p.get('path') for p in defn_parts]}")
        elif vr.status_code == 202:
            # LRO for getDefinition
            loc = vr.headers.get("Location", "")
            if loc:
                for _ in range(15):
                    time.sleep(2)
                    pr = requests.get(loc, headers=hdrs)
                    if pr.status_code == 200:
                        vdata = pr.json()
                        defn_parts = vdata.get("definition", {}).get("parts", [])
                        content_part = next((p for p in defn_parts if "notebook-content" in p.get("path", "")), None)
                        if content_part:
                            decoded = base64.b64decode(content_part["payload"]).decode("utf-8")
                            line_count = decoded.count("\n")
                            print(f"  {test_nb}: {line_count} lines — {'OK' if line_count > 10 else 'EMPTY!'}")
                        else:
                            print(f"  WARNING: No notebook-content part in definition")
                        break
                else:
                    print(f"  Verification timed out")
        else:
            print(f"  Verification failed: HTTP {vr.status_code}")
    else:
        print(f"  Cannot verify — {test_nb} not found in workspace")
    print(f"{'='*60}")

In [ ]:
# ============================================================================
# CELL 4 — Upload healthcare knowledge docs to Data Agent lakehouse
# ============================================================================

if UPLOAD_KNOWLEDGE_DOCS:
    import os, glob

    # fabric-launcher extracts the repo ZIP to this relative path by default
    # Try both relative (library default) and absolute (Fabric mount) paths
    for candidate in [".lakehouse/default/Files/src", "/lakehouse/default/Files/src"]:
        if os.path.isdir(candidate):
            extract_base = candidate
            break
    else:
        print("Warning: Could not find extract directory. Skipping knowledge doc upload.")
        print("  Tried: .lakehouse/default/Files/src and /lakehouse/default/Files/src")
        UPLOAD_KNOWLEDGE_DOCS = False
        extract_base = None

    # fabric-launcher extracts repo contents directly into extract_base (no wrapper folder)
    repo_root = extract_base

    knowledge_src = os.path.join(repo_root, "healthcare_knowledge") if repo_root else None

    if knowledge_src and os.path.exists(knowledge_src):
        print("Uploading healthcare knowledge documents...")
        launcher.upload_files_to_lakehouse(
            lakehouse_name="lh_gold_curated",
            source_directory=knowledge_src,
            target_folder="healthcare_knowledge",
            file_patterns=["*.md"],
        )
        # Count uploaded files
        count = sum(len(files) for _, _, files in os.walk(knowledge_src) if files)
        print(f"Uploaded {count} knowledge documents to lh_gold_curated/Files/healthcare_knowledge/")
    else:
        print(f"Warning: healthcare_knowledge folder not found at {knowledge_src}")
else:
    print("Skipping knowledge doc upload (UPLOAD_KNOWLEDGE_DOCS=False)")

In [ ]:
# ============================================================================
# CELL 5 — Generate synthetic data
# ============================================================================

if GENERATE_DATA:
    print("Running NB_Generate_Sample_Data (generates ~10K patients, 100K encounters)...")
    print("This takes 2-4 minutes.\n")

    result = launcher.run_notebook_synchronous(
        notebook_name="NB_Generate_Sample_Data",
        parameters={},
        timeout_seconds=1200,  # 20 min max
    )

    if result.get("success"):
        print("\nData generation SUCCEEDED")
    else:
        status = result.get("status", "Unknown")
        print(f"\nData generation returned status: {status}")
        print("Check the NB_Generate_Sample_Data notebook for details.")
else:
    print("Skipping data generation (GENERATE_DATA=False)")

In [ ]:
# ============================================================================
# CELL 6 — Run the full-load pipeline
# ============================================================================

if RUN_PIPELINE:
    import requests, time, json

    token = notebookutils.credentials.getToken("pbi")
    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
    api_base = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}"

    # Find the pipeline ID
    print("Looking up PL_Healthcare_Master pipeline...")
    resp = requests.get(f"{api_base}/items?type=DataPipeline", headers=headers)
    resp.raise_for_status()
    pipelines = resp.json().get("value", [])
    pipeline = next((p for p in pipelines if p["displayName"] == "PL_Healthcare_Master"), None)

    if not pipeline:
        print("WARNING: PL_Healthcare_Master not found. Skipping pipeline run.")
        print("You can run it manually from the workspace.")
    else:
        pipeline_id = pipeline["id"]
        print(f"Pipeline ID: {pipeline_id}")

        # Trigger pipeline with load_mode=full
        print("Triggering pipeline with load_mode=full...")
        trigger_body = {
            "executionData": {
                "parameters": {
                    "load_mode": "full"
                }
            }
        }
        resp = requests.post(
            f"{api_base}/items/{pipeline_id}/jobs/instances?jobType=Pipeline",
            headers=headers,
            json=trigger_body,
        )

        if resp.status_code in (200, 202):
            # Get job ID from Location header or response
            location = resp.headers.get("Location", "")
            print(f"Pipeline triggered. Polling for completion...")
            print(f"(This takes 8-15 minutes for full load)\n")

            # Poll until complete
            max_polls = 120  # 120 * 15s = 30 min max
            for poll in range(max_polls):
                time.sleep(15)
                try:
                    if location:
                        poll_resp = requests.get(location, headers=headers)
                    else:
                        poll_resp = requests.get(
                            f"{api_base}/items/{pipeline_id}/jobs/instances",
                            headers=headers,
                        )
                    if poll_resp.status_code == 200:
                        job_data = poll_resp.json()
                        status = job_data.get("status", "Unknown")
                        if status in ("Completed", "Succeeded"):
                            print(f"  Pipeline COMPLETED after {(poll+1)*15}s")
                            break
                        elif status in ("Failed", "Cancelled"):
                            print(f"  Pipeline {status} after {(poll+1)*15}s")
                            print(f"  Check the pipeline run history for details.")
                            break
                        else:
                            if poll % 4 == 0:  # Print every 60s
                                print(f"  [{(poll+1)*15}s] Status: {status}...")
                except Exception as e:
                    if poll % 4 == 0:
                        print(f"  [{(poll+1)*15}s] Polling... ({e})")
            else:
                print("  Pipeline still running after 30 min. Check workspace for status.")
        else:
            print(f"  Pipeline trigger returned {resp.status_code}: {resp.text}")
            print("  You can run it manually from the workspace.")
else:
    print("Skipping pipeline run (RUN_PIPELINE=False)")
    print("To run manually: Open PL_Healthcare_Master → Run with load_mode=full")

In [ ]:
# ============================================================================
# CELL 7 — Ontology & Graph Model (Manual Setup Required)
# ============================================================================
# The Fabric API cannot fully deploy an ontology + graph as LINKED items.
# Creating via API results in unlinked ontology and graph — which breaks
# Fabric IQ graph traversal and Copilot integration.
#
# Instead, you create it from the semantic model in the UI (takes ~10 min).
# A detailed step-by-step guide is included in this repo.
# ============================================================================

print("=" * 60)
print("  ONTOLOGY & GRAPH MODEL — Manual Setup")
print("=" * 60)
print()
print("  The ontology must be created in the Fabric UI to ensure")
print("  the ontology and graph model are properly LINKED.")
print()
print("  Follow the step-by-step guide:")
print("  📄 ONTOLOGY_GRAPH_SETUP_GUIDE.md (in this repo)")
print()
print("  Quick summary:")
print("  1. New item → Ontology → from semantic model 'HealthcareDemoHLS'")
print("  2. Delete 3 unwanted entities (dim_date, agg_medication_adherence, agg_readmission_by_date)")
print("  3. Rename 10 entities (e.g., dim_patient → Patient)")
print("  4. Set source keys and display names per the guide's table")
print("  5. Delete auto-generated relationships, add 15 curated ones")
print("  6. Build the graph (Graph tab → Build a graph → select all)")
print()
print("  The guide includes:")
print("  - Master entity configuration table (10 entities)")
print("  - Master relationship table (15 relationships)")
print("  - Full property reference for all entity types")
print("  - Verification steps")
print()
print("  ⏱️  Estimated time: 10-15 minutes")
print("=" * 60)

In [ ]:
# ============================================================================
# CELL 7b — Deploy Real-Time Intelligence (RTI)
# ============================================================================
# Eventhouse + KQL Database are already deployed as Git artifacts (Stage 2).
# This cell runs post-deploy wiring and scoring notebooks:
#   1. NB_RTI_Setup_Eventhouse  — Discovers artifacts, creates tables, creates Eventstream
#   2. NB_RTI_Event_Simulator   — Generates initial events (batch mode)
#   3. NB_RTI_Fraud_Detection   — Scores claims for fraud patterns
#   4. NB_RTI_Care_Gap_Alerts   — Point-of-care gap closure alerts
#   5. NB_RTI_HighCost_Trajectory — High-cost member trajectory analysis
# ============================================================================

if DEPLOY_RTI:
    print("=" * 60)
    print("  REAL-TIME INTELLIGENCE DEPLOYMENT")
    print("=" * 60)

    rti_notebooks = [
        ("NB_RTI_Setup_Eventhouse", {"WORKSPACE_ID": workspace_id}),
        ("NB_RTI_Event_Simulator", {"MODE": "batch", "BATCH_SIZE": "500"}),
        ("NB_RTI_Fraud_Detection", {}),
        ("NB_RTI_Care_Gap_Alerts", {}),
        ("NB_RTI_HighCost_Trajectory", {}),
    ]

    for nb_name, params in rti_notebooks:
        print(f"\n  Running {nb_name}...")
        try:
            result = launcher.run_notebook_synchronous(
                notebook_name=nb_name,
                parameters=params,
                timeout_seconds=1200,
            )
            status = "OK" if result.get("success") else result.get("status", "???")
            print(f"  -> {nb_name}: {status}")
        except Exception as e:
            print(f"  -> {nb_name}: FAILED -- {e}")
            print(f"    You can run this notebook manually from the workspace.")

    print("\n" + "=" * 60)
    print("  RTI DEPLOYMENT COMPLETE")
    print("=" * 60)
    print("  Tables created in lh_gold_curated:")
    print("    - rti_claims_events, rti_adt_events, rti_rx_events")
    print("    - rti_fraud_scores, rti_care_gap_alerts, rti_highcost_alerts")
    print()
    print("  Eventhouse: Healthcare_RTI_Eventhouse (+ KQL DB)")
    print("  For streaming mode: set MODE='stream' in NB_RTI_Event_Simulator")
    print("=" * 60)
else:
    print("Skipping RTI deployment (DEPLOY_RTI=False)")

In [ ]:
# ============================================================================
# CELL 8 — Deployment Summary
# ============================================================================

import requests

token = notebookutils.credentials.getToken("pbi")
headers = {"Authorization": f"Bearer {token}"}
api_base = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}"

resp = requests.get(f"{api_base}/items", headers=headers)
items = resp.json().get("value", []) if resp.status_code == 200 else []

# Group by type
by_type = {}
for item in items:
    t = item.get("type", "Unknown")
    by_type.setdefault(t, []).append(item["displayName"])

print("=" * 60)
print("  DEPLOYMENT COMPLETE")
print("=" * 60)
print(f"  Workspace: {workspace_id}")
print(f"  Total items: {len(items)}")
print()
for item_type in ["Lakehouse", "Notebook", "DataPipeline", "SemanticModel", "DataAgent", "Ontology", "Eventhouse", "KQLDatabase", "Eventstream", "SQLEndpoint"]:
    names = by_type.get(item_type, [])
    if names:
        print(f"  {item_type} ({len(names)}):")
        for n in sorted(names):
            print(f"    - {n}")

# Show any other types
shown = {"Lakehouse", "Notebook", "DataPipeline", "SemanticModel", "DataAgent", "Ontology", "Eventhouse", "KQLDatabase", "Eventstream", "SQLEndpoint"}
for item_type, names in by_type.items():
    if item_type not in shown:
        print(f"  {item_type} ({len(names)}):")
        for n in sorted(names):
            print(f"    - {n}")

# RTI table counts
print()
print("  RTI Delta Tables (lh_gold_curated):")
try:
    for tbl in ["rti_claims_events", "rti_adt_events", "rti_rx_events", "rti_fraud_scores", "rti_care_gap_alerts", "rti_highcost_alerts"]:
        try:
            cnt = spark.table(tbl).count()
            print(f"    - {tbl}: {cnt:,} rows")
        except Exception:
            print(f"    - {tbl}: (not created yet)")
except Exception:
    print("    (could not query RTI tables)")

print()
print("=" * 60)
print("  NEXT STEPS")
print("=" * 60)
print("  1. Open the HealthcareDemoHLS semantic model -> verify tables loaded")
print("  2. Open the HealthcareHLSAgent -> ask: 'What are the top denial reasons?'")
print("  3. Create the Ontology + Graph Model in the Fabric UI")
print("     -> Follow ONTOLOGY_GRAPH_SETUP_GUIDE.md (~10 min)")
print("  4. For incremental loads: run NB_Generate_Incremental_Data, then")
print("     run PL_Healthcare_Master with load_mode=incremental")
print("  5. RTI: Open Healthcare_RTI_Eventhouse to explore real-time data")
print("     - NB_RTI_Fraud_Detection: View rti_fraud_scores for flagged claims")
print("     - NB_RTI_Care_Gap_Alerts: View rti_care_gap_alerts for gap closures")
print("     - NB_RTI_HighCost_Trajectory: View rti_highcost_alerts for cost trends")
print("  6. For live streaming: set MODE='stream' in NB_RTI_Event_Simulator")
print("     and configure the Eventstream Custom App connection string")
print("=" * 60)